In [ ]:
from datetime import datetime
import glob
import medspacy
from medspacy.section_detection import Sectionizer
from medspacy.section_detection import SectionRule
import os
import pandas as pd
import re
from tqdm import tqdm

In [ ]:
#set date and output drive
today = datetime.today().strftime('%Y%m%d')

In [ ]:
#identify categories we're interested in
section_categories = set([
                          'chief_complaint',
                          'history_of_present_illness'
                          'social_history',
                          'medical_decision_making',
                          'observation_and_plan',
                          None
                         ])
#section_categories.add(None)

In [ ]:
#Implement custom sectionizer rules
custom_rules = [
    # See Custom_Sectionizer_Rules.ipynb
]

for rule in custom_rules:
    rule.pattern_lower = False
    rule.pattern_type = "literal"

print(f"{len(custom_rules)} Custom Rules")

In [ ]:
def extract_interesting_sections(docs, section_categories):
    # Tag all text matching headers defined in Custom_Sectionizer_Rules with standardized section_category.
    # Keep only section categories in the set section_categories defined previously.
    
    output = []
    for i, doc in enumerate(tqdm(docs, desc="Extracting Sections")):
        section_texts = []
        for section in doc._.sections:
            if section.category in section_categories:
                section_body = section.body_span
                if section_body:
                    section_text = doc[section_body[0]:section_body[1]].text
                    section_texts.append(section_text)
        if section_texts:
            full_doc = "\n\n".join(section_texts)
            output.append(full_doc)
    return output

In [ ]:
def word_count(txt: str) -> int:
    return len(re.findall(r"\S+", txt))

In [ ]:
# Initialize sectionizer pipeline.
nlp = medspacy.load()
sectionizer = nlp.add_pipe("medspacy_sectionizer", config={"phrase_matcher_attr": "TEXT"})
sectionizer.add(custom_rules)

In [ ]:
# Identify start dates for each outbreak in SSMS table using SQL query.
case_names = ['COVID19', 'Leptospirosis', 'Mpox', 'Zika']

initial_start_dates = []

for case_name in case_names:
    query = f"""
    SELECT *
    FROM [Table]
    WHERE DiseaseCol = '{case_name}'
    """

    df = pd.read_sql(query, conn)
    start_date = df['CaseDate'].min()
    initial_start_dates.append(start_date)

initial_params = list(zip(case_names, initial_start_dates))
print(initial_params)

In [ ]:
#Sectionize documents from the identified start date through the first six months of each outbreak.
#Pull the first 3 for each encounter with >= 3 documents.

initial_results = {}

for case_name, initial_date in initial_params:
    query = f"""/****** Script for SelectTopNRows command from SSMS  ******/
    WITH ranked_docs AS (
    	SELECT *
    	  ,ROW_NUMBER() OVER(
    		PARTITION BY VisitCol
    		ORDER BY TimeCol
    	) AS rn
    	FROM [Target_Table]
    	WHERE DocCol NOT LIKE '%educat%'
    	AND DocCol NOT LIKE '%nur%'
    	AND DocCol NOT LIKE '%rn %'
        AND DocCol NOT LIKE '%NSG%' 
    	AND DocCol NOT LIKE '%reconcil%'
    	AND DocCol NOT LIKE '%dispo%'
    	AND DocCol NOT LIKE '%transfer%'
    	AND DocCol NOT LIKE '%pharmacy%'
    	AND DocCol NOT LIKE '%consent%'
    	AND DocCol NOT LIKE '%psych%'
    	AND DocCol NOT LIKE '%nutrition%'
    	AND DocCol NOT LIKE '%discharge%'
    	AND DocCol NOT LIKE '%pain%'
    	AND DocCol NOT LIKE '%admission screen%'
    	AND DocCol NOT LIKE '%admission assessment%'
    	AND DocCol NOT LIKE '%admission evaluation%'
        AND DocCol NOT LIKE '%10-0386%'
        AND DocCol NOT LIKE '%FLOWSHEET%'
        AND DocCol NOT LIKE '%OUTPUT%'
        AND DocCol NOT LIKE '%dental%'
        AND DocCol NOT LIKE '%homeless%'
        AND DocCol NOT LIKE '%I&O%'
        AND DocCol NOT LIKE '%AUTO PRINT%'
        AND DocCol NOT LIKE '%AFTERCARE%'
        AND DocCol NOT LIKE '%TRIAGE%'
        AND DocCol NOT LIKE '%SURG%'
        AND DocCol NOT LIKE '%PROGRESS%'
        AND DocCol NOT LIKE '%FALL RISK%'
        AND DocCol NOT LIKE '%SPIRITUAL%'
        AND DocCol NOT LIKE '%OUTPT%'
        AND DocCol NOT LIKE '%OUTPATIENT%'
        AND DocCol NOT LIKE '%DISCH%'
        AND DocCol NOT LIKE '%DISCH%'
        AND DocCol NOT LIKE '%SKIN%'
        AND DocCol NOT LIKE '%CLC%'
        AND DocCol NOT LIKE '%OPERATION REPORT%'
        AND DocCol NOT LIKE 'PATHOLOGY REPORT%'
        AND DocCol NOT LIKE '%INTRAOPERATIVE REPORT%'
        AND DocCol NOT LIKE '%SOCIAL WORK%'
        AND DocCol NOT LIKE '%COMMUNITY CARE%'
    )
    SELECT *
    FROM ranked_docs
    WHERE rn <= 3
    	AND DiseaseCol LIKE '%{case_name}%'
    	AND CaseDateCol BETWEEN '{initial_date}' AND DATEADD(MONTH, 6, '{initial_date}')
    ORDER BY CaseDateCol ASC;
    
    """
    
    df = pd.read_sql(query, conn)
    df = df.dropna(subset=['TextCol']).drop_duplicates(subset=['TextCol'])
    df['TextCol'] = df['TextCol'].astype(str)

    case_names = df['DiseaseCol'].unique().tolist()
    print(case_names)
    
    for case_name in case_names:
        EHR_records = (df.loc[df['DiseaseCol'] == case_name, 'TextCol']
                       .dropna()
                       .astype(str).
                       drop_duplicates()
                       .tolist())
        
        docs = list(tqdm(nlp.pipe(EHR_records, batch_size=12),
                         total=len(EHR_records),
                         desc=f"Sectionizing {case_name} EHRs"))
        cleaned_EHRs = extract_interesting_sections(docs, section_categories)
        
        filtered_EHRs = [t for t in cleaned_EHRs if word_count(t) >= 50]
    
        initial_results[case_name] = "\n---END OF EHR---\n".join(filtered_EHRs)

## Sorting Random and Syndrome EHRs per disease

This merges all the EHRs not considered 'case' and not considered 'syndrome' into RandomVisits.
"---END OF EHR---" is the delimeter that will be used by LIWC-22 to partition individual documents.

Syndrome Definitions:

- COVID19: Resp-Pneumonia
- Leptospirosis: Rash & Conjunctivitis
- Mpox: Rash
- Zika: Rash

In [ ]:
#COVID19
initial_results["AllRandomVisits_COVID19"] = (initial_results.pop("RandomVisits_COVID19") + "\n\n---END OF EHR---\n\n " +
                                              initial_results.pop("RandomVisits_COVID19_ISDS_ICD10_Conjunctivitis") + "\n\n---END OF EHR---\n\n " +
                                              initial_results.pop("RandomVisits_COVID19_ISDS_ICD10_Gastrointestinal") + "\n\n---END OF EHR---\n\n " +
                                              initial_results.pop("RandomVisits_COVID19_ISDS_ICD10_Rash")
                                             )

initial_results["Syndrome_COVID19"] = initial_results.pop("RandomVisits_COVID19_ISDS_ICD10_Resp-Pneumonia")

initial_results["TrueCase_COVID19"] = initial_results.pop("COVID19")

#Lepto
initial_results["AllRandomVisits_Lepto"] = (initial_results.pop("RandomVisits_Leptospirosis") + "\n\n---END OF EHR---\n\n " +
                                                    initial_results.pop("RandomVisits_Leptospirosis_ISDS_ICD10_Gastrointestinal") + "\n\n---END OF EHR---\n\n " +
                                                    initial_results.pop("RandomVisits_Leptospirosis_ISDS_ICD10_Resp-Pneumonia")
                                                   )

initial_results["Syndrome_Lepto"] = (initial_results.pop("RandomVisits_Leptospirosis_ISDS_ICD10_Conjunctivitis") + "\n\n---END OF EHR---\n\n " +
                                             initial_results.pop("RandomVisits_Leptospirosis_ISDS_ICD10_Rash"))

initial_results["TrueCase_Lepto"] = initial_results.pop("Leptospirosis")

#Mpox
initial_results["AllRandomVisits_Mpox"] = (initial_results.pop("RandomVisits_Mpox") + "\n\n---END OF EHR---\n\n " +
                                            initial_results.pop("RandomVisits_Mpox_ISDS_ICD10_Conjunctivitis") + "\n\n---END OF EHR---\n\n " +
                                            initial_results.pop("RandomVisits_Mpox_ISDS_ICD10_Gastrointestinal") + "\n\n---END OF EHR---\n\n " +
                                            initial_results.pop("RandomVisits_Mpox_ISDS_ICD10_Resp-Pneumonia"))

initial_results["Syndrome_Mpox"] = initial_results.pop("RandomVisits_Mpox_ISDS_ICD10_Rash")

initial_results["TrueCase_Mpox"] = initial_results.pop("Mpox")

#Zika
initial_results["AllRandomVisits_Zika"] = (initial_results.pop("RandomVisits_Zika") + "\n\n---END OF EHR---\n\n " +
                                            initial_results.pop("RandomVisits_Zika_ISDS_ICD10_Conjunctivitis") + "\n\n---END OF EHR---\n\n " +
                                            initial_results.pop("RandomVisits_Zika_ISDS_ICD10_Gastrointestinal") + "\n\n---END OF EHR---\n\n " +
                                            initial_results.pop("RandomVisits_Zika_ISDS_ICD10_Resp-Pneumonia")
                                            )

initial_results["Syndrome_Zika"] = initial_results.pop("RandomVisits_Zika_ISDS_ICD10_Rash")

initial_results["TrueCase_Zika"] = initial_results.pop("Zika")

In [ ]:
initial_results["AllRandomVisits_Combined"] = initial_results['AllRandomVisits_COVID19'] + initial_results['AllRandomVisits_Lepto'] + initial_results['AllRandomVisits_Mpox'] + initial_results['AllRandomVisits_Zika']
initial_results["Syndrome_Combined"] = initial_results['Syndrome_COVID19'] + initial_results['Syndrome_Lepto'] + initial_results['Syndrome_Mpox'] + initial_results['Syndrome_Zika']
initial_results["TrueCase_Combined"] = initial_results['TrueCase_COVID19'] + initial_results['TrueCase_Lepto'] + initial_results['TrueCase_Mpox'] + initial_results['TrueCase_Zika']

In [ ]:
#sectionize from late time period

late_results = {}

for case_name, initial_date in initial_params:
    query = f"""/****** Script for SelectTopNRows command from SSMS  ******/
    WITH ranked_docs AS (
    	SELECT *
    	  ,ROW_NUMBER() OVER(
    		PARTITION BY VisitCol
    		ORDER BY CaseDateCol
    	) AS rn
    	FROM [TABLE]
    	WHERE DocCol NOT LIKE '%educat%'
        
    	AND DocCol NOT LIKE '%nur%'
    	AND DocCol NOT LIKE '%rn %'
        AND DocCol NOT LIKE '%NSG%'
        
    	AND DocCol NOT LIKE '%reconcil%'
    	AND DocCol NOT LIKE '%dispo%'
    	AND DocCol NOT LIKE '%transfer%'
    	AND DocCol NOT LIKE '%pharmacy%'
    	AND DocCol NOT LIKE '%consent%'
    	AND DocCol NOT LIKE '%psych%'
    	AND DocCol NOT LIKE '%nutrition%'
    	AND DocCol NOT LIKE '%discharge%'
    	AND DocCol NOT LIKE '%pain%'
    	AND DocCol NOT LIKE '%admission screen%'
    	AND DocCol NOT LIKE '%admission assessment%'
    	AND DocCol NOT LIKE '%admission evaluation%'
        AND DocCol NOT LIKE '%10-0386%'
        AND DocCol NOT LIKE '%FLOWSHEET%'
        AND DocCol NOT LIKE '%OUTPUT%'
        AND DocCol NOT LIKE '%dental%'
        AND DocCol NOT LIKE '%homeless%'
        AND DocCol NOT LIKE '%I&O%'
        AND DocCol NOT LIKE '%AUTO PRINT%'
        AND DocCol NOT LIKE '%AFTERCARE%'
        AND DocCol NOT LIKE '%TRIAGE%'
        AND DocCol NOT LIKE '%SURG%'
        AND DocCol NOT LIKE '%PROGRESS%'
        AND DocCol NOT LIKE '%FALL RISK%'
        AND DocCol NOT LIKE '%SPIRITUAL%'
        AND DocCol NOT LIKE '%OUTPT%'
        AND DocCol NOT LIKE '%OUTPATIENT%'
        AND DocCol NOT LIKE '%DISCH%'
        AND DocCol NOT LIKE '%DISCH%'
        AND DocCol NOT LIKE '%SKIN%'
        AND DocCol NOT LIKE '%CLC%'
        AND DocCol NOT LIKE '%OPERATION REPORT%'
        AND DocCol NOT LIKE 'PATHOLOGY REPORT%'
        AND DocCol NOT LIKE '%INTRAOPERATIVE REPORT%'
        AND DocCol NOT LIKE '%SOCIAL WORK%'
        AND DocCol NOT LIKE '%COMMUNITY CARE%'
    )
    SELECT *
    FROM ranked_docs
    WHERE rn <= 3
    	AND DiseaseCol LIKE '%{case_name}%'
    	AND CaseDateCol >= DATEADD(YEAR, 1, '{initial_date}')
    ORDER BY CaseDateCol ASC;
    
    """
    
    #list report text
    df = pd.read_sql(query, conn)
    df = df.dropna(subset=['TextCol']).drop_duplicates(subset=['TextCol'])
    df['TextCol'] = df['TextCol'].astype(str)
    
    case_names = df['DiseaseCol'].unique().tolist()
    print(case_names)
    
    for case_name in case_names:
        EHR_records = (df.loc[df['DiseaseCol'] == case_name, 'TextCol']
                       .dropna()
                       .astype(str).
                       drop_duplicates()
                       .tolist())
        
        docs = list(tqdm(nlp.pipe(EHR_records, batch_size=12),
                         total=len(EHR_records),
                         desc=f"Sectionizing {case_name} EHRs"))
        cleaned_EHRs = extract_interesting_sections(docs, section_categories)
        
        filtered_EHRs = [t for t in cleaned_EHRs if word_count(t) >= 50]
        
        late_results[case_name] = "\n---END OF EHR---\n".join(filtered_EHRs)

In [ ]:
#COVID19
late_results["AllRandomVisits_COVID19"] = (late_results.pop("RandomVisits_COVID19") + "\n\n---END OF EHR---\n\n " +
    late_results.pop("RandomVisits_COVID19_ISDS_ICD10_Conjunctivitis") + "\n\n---END OF EHR---\n\n " +
    late_results.pop("RandomVisits_COVID19_ISDS_ICD10_Gastrointestinal") + "\n\n---END OF EHR---\n\n " +
    late_results.pop("RandomVisits_COVID19_ISDS_ICD10_Rash")
                                             )

late_results["Syndrome_COVID19"] = late_results.pop("RandomVisits_COVID19_ISDS_ICD10_Resp-Pneumonia")

late_results["TrueCase_COVID19"] = late_results.pop("COVID19")

#Lepto
late_results["AllRandomVisits_Lepto"] = (late_results.pop("RandomVisits_Leptospirosis") + "\n\n---END OF EHR---\n\n " +
    late_results.pop("RandomVisits_Leptospirosis_ISDS_ICD10_Gastrointestinal") + "\n\n---END OF EHR---\n\n " +
    late_results.pop("RandomVisits_Leptospirosis_ISDS_ICD10_Resp-Pneumonia")
                                                   )

late_results["Syndrome_Lepto"] = (late_results.pop("RandomVisits_Leptospirosis_ISDS_ICD10_Conjunctivitis") + "\n\n---END OF EHR---\n\n " +
                                             late_results.pop("RandomVisits_Leptospirosis_ISDS_ICD10_Rash")
                                            )

late_results["TrueCase_Lepto"] = late_results.pop("Leptospirosis")

#Mpox
late_results["AllRandomVisits_Mpox"] = (late_results.pop("RandomVisits_Mpox") + "\n\n---END OF EHR---\n\n " +
    late_results.pop("RandomVisits_Mpox_ISDS_ICD10_Conjunctivitis") + "\n\n---END OF EHR---\n\n " +
    late_results.pop("RandomVisits_Mpox_ISDS_ICD10_Gastrointestinal") + "\n\n---END OF EHR---\n\n " +
    late_results.pop("RandomVisits_Mpox_ISDS_ICD10_Resp-Pneumonia")
                                          )

late_results["Syndrome_Mpox"] = late_results.pop("RandomVisits_Mpox_ISDS_ICD10_Rash")

late_results["TrueCase_Mpox"] = late_results.pop("Mpox")

#Zika
late_results["AllRandomVisits_Zika"] = (late_results.pop("RandomVisits_Zika") + "\n\n---END OF EHR---\n\n " +
    late_results.pop("RandomVisits_Zika_ISDS_ICD10_Conjunctivitis") + "\n\n---END OF EHR---\n\n " +
    late_results.pop("RandomVisits_Zika_ISDS_ICD10_Gastrointestinal") + "\n\n---END OF EHR---\n\n " +
    late_results.pop("RandomVisits_Zika_ISDS_ICD10_Resp-Pneumonia")
                                          )

late_results["Syndrome_Zika"] = late_results.pop("RandomVisits_Zika_ISDS_ICD10_Rash")

late_results["TrueCase_Zika"] = late_results.pop("Zika")

In [ ]:
late_results["AllRandomVisits_Combined"] = late_results['AllRandomVisits_COVID19'] + late_results['AllRandomVisits_Lepto'] + late_results['AllRandomVisits_Mpox'] + late_results['AllRandomVisits_Zika']
late_results["Syndrome_Combined"] = late_results['Syndrome_COVID19'] + late_results['Syndrome_Lepto'] + late_results['Syndrome_Mpox'] + late_results['Syndrome_Zika']
late_results["TrueCase_Combined"] = late_results['TrueCase_COVID19'] + late_results['TrueCase_Lepto'] + late_results['TrueCase_Mpox'] + late_results['TrueCase_Zika']

In [ ]:
print("Initial Case Counts:")
for key, content in initial_results.items():
    print(f"{key}, {len(content.split('---END OF EHR---'))}")

In [ ]:
print("Late Case Counts:")
for key, content in late_results.items():
    print(f"{key}, {len(content.split('---END OF EHR---'))}")

## Save .txt files for LIWC-22 analysis

In [ ]:
dir_path = fr"output_dir\{today}"
os.makedirs(dir_path, exist_ok=True)

for key, content in initial_results.items():
    file_path = os.path.join(dir_path, f"{key}_Initial.txt")
    
    with open(file_path, "w", encoding="utf-8") as f:
        f.write(content)

for key, content in late_results.items():
    file_path = os.path.join(dir_path, f"{key}_Late.txt")
    
    with open(file_path, "w", encoding="utf-8") as f:
        f.write(content)